# Import packages

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os, sys

# Ensure project root
for p in [Path.cwd()] + list(Path.cwd().parents):
    if p.name == 'Multifirefly-Project':
        os.chdir(p)
        sys.path.insert(0, str(p / 'multiff_analysis/multiff_code/methods'))
        break

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rc

from data_wrangling import general_utils
from visualization.matplotlib_tools import additional_plots
from visualization.animation import animation_class
from reinforcement_learning.agents.feedforward import sb3_class, sb3_env
from reinforcement_learning.collect_data import process_agent_data
from null_behaviors import show_null_trajectory

plt.rcParams["animation.html"] = "html5"
rc('animation', html='jshtml')
pd.set_option('display.float_format', lambda x: '%.5f' % x)
np.set_printoptions(suppress=True)
pd.options.display.max_rows = 101
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# load monkey data

In [ ]:
PLAYER = "monkey"
raw_data_folder_path = "all_monkey_data/raw_monkey_data/monkey_Bruno/data_0330"

data_item = animation_class.AnimationClass(
    raw_data_folder_path=raw_data_folder_path
)

data_item.retrieve_or_make_monkey_data(
    exists_ok=True,
    min_distance_to_calculate_angle=10
)

data_item.make_or_retrieve_ff_dataframe(exists_ok=True)

data_item.make_info_of_monkey()


# load RL agent

In [ ]:
model_folder_name = (
    "RL_models/sb3_stored_models/all_agents/"
    "agents_with_noise/ff6_mem3_drop_fill_visible_only/"
    "vn0_wn0_pr0p0005_pt0p005_mr0p002_mt0p002/"
)

agent = sb3_class.SB3forMultifirefly(
    model_folder_name=model_folder_name
)
agent.load_latest_agent()

env_kwargs = agent.input_env_kwargs.copy()
agent.env_for_data_collection = sb3_env.CollectInformation(**env_kwargs)


# Plot RL & monkey side_by_side

In [ ]:
agent_dt = 0.25
num_trials = 2

plotting_params = {
    "show_stops": True,
    "show_believed_target_positions": True,
    "show_reward_boundary": True,
    "show_scale_bar": True,
    "hitting_arena_edge_ok": True,
    "trial_too_short_ok": True
}


# =====================
# Run Selected Trial
# =====================

for currentTrial in [496]:

    ff_caught = data_item.info_of_monkey['ff_caught_T_new']

    start_time = min(
        ff_caught[currentTrial - 3],
        ff_caught[currentTrial] - num_trials
    )

    whole_plot_duration = [start_time, ff_caught[currentTrial]]
    imitation_duration = [start_time, ff_caught[currentTrial] - 1.5]

    alive_ffs = np.where(
        (data_item.info_of_monkey['ff_life_sorted'][:, 1] >= whole_plot_duration[0]) &
        (data_item.info_of_monkey['ff_life_sorted'][:, 0] < whole_plot_duration[1])
    )[0]

    env_kwargs.update({
        'arena_radius': 1000,
        'num_alive_ff': len(alive_ffs),
        'recentering_trigger_radius': 10000
    })

    agent.env_for_data_collection = sb3_env.CollectInformation(**env_kwargs)

    info_of_agent, rotation_matrix, \
    num_imitation_steps_monkey, num_imitation_steps_agent = (
        process_agent_data.find_corresponding_info_of_agent(
            agent.env_for_data_collection,
            agent.rl_agent,
            data_item.info_of_monkey,
            start_time,
            whole_plot_duration,
            imitation_duration,
        )
    )

    info_of_agent['currentTrial'] = currentTrial
    info_of_agent['num_trials'] = num_trials

    with general_utils.initiate_plot(20, 20, 400):

        additional_plots.PlotSidebySide(
            whole_duration=whole_plot_duration,
            info_of_monkey=data_item.info_of_monkey,
            info_of_agent=info_of_agent,
            num_imitation_steps_monkey=num_imitation_steps_monkey,
            num_imitation_steps_agent=num_imitation_steps_agent,
            currentTrial=currentTrial,
            num_trials=num_trials,
            rotation_matrix=rotation_matrix,
            plotting_params=plotting_params,
        )